In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import torch 
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset , DataLoader

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
device ="cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df=pd.read_csv('/kaggle/input/digit-recognizer/train.csv')

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df=train_test_split(df,train_size=0.8,random_state=42)
train_df=train_df.to_numpy()
test_df=test_df.to_numpy()
y_train, x_train = train_df[:,0], train_df[:,1:].astype(np.float32)
y_test,x_test=test_df[:,0], test_df[:,1:].astype(np.float32)

In [ ]:
x_train,y_train,x_test,y_test=map(torch.tensor,(x_train,y_train,x_test,y_test))

In [ ]:
eps=1e-6
def relu(x):
    return x.clamp(min=0)

def batch_norm(x,gamma):
    batch_mean=x.mean(dim=0)
    batch_var=(x-batch_mean).var(dim=0)
    x_norm=(x-batch_mean) / torch.sqrt(batch_var + eps)
    return gamma*x_norm

In [ ]:
BATCH_SIZE=64
epochs=300
EPOCHS_PER_EVAL=10
lr=1e-2
hidden_features=512

In [ ]:
x_train,y_train,x_test,y_test=[t.to(device) for t in [x_train,y_train,x_test,y_test]]
w1=torch.distributions.Uniform(-np.sqrt(1/784),np.sqrt(1/784)).sample((hidden_features,784)).to(device).requires_grad_()
b1=torch.distributions.Uniform(-np.sqrt(1/784),np.sqrt(1/784)).sample((hidden_features,)).to(device).requires_grad_()
w2=torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((hidden_features,hidden_features)).to(device).requires_grad_()
b2=torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((hidden_features,)).to(device).requires_grad_()
w3=torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((10,hidden_features)).to(device).requires_grad_()
b3=torch.distributions.Uniform(-np.sqrt(1/hidden_features),np.sqrt(1/hidden_features)).sample((10,)).to(device).requires_grad_()

In [ ]:
class MNIST:
    def __init__(self):
        self.W1 = (torch.rand(256, 784) - 0.5)/torch.sqrt(torch.tensor(784))
        self.b1 = (torch.rand(256) - 0.5)/torch.sqrt(torch.tensor(784))
        self.W2 = (torch.rand(100, 256) - 0.5)/torch.sqrt(torch.tensor(256))
        self.b2 = (torch.rand(100) - 0.5)/torch.sqrt(torch.tensor(256))
        self.W3 = (torch.rand(10, 100) - 0.5)/torch.sqrt(torch.tensor(100))
        self.b3 = (torch.rand(10) - 0.5)/torch.sqrt(torch.tensor(100))
        self.W1.to(device).requires_grad_()
        self.b1.to(device).requires_grad_()
        self.W2.to(device).requires_grad_()
        self.b2.to(device).requires_grad_()
        self.W3.to(device).requires_grad_()
        self.b3.to(device).requires_grad_()

    def __call__(self, x):
        y1 = relu(x@self.W1.t() + self.b1)
        y2 = relu(y1@self.W2.t() + self.b2)
        logits = y2@self.W3.t() + self.b3
        return logits

In [ ]:
def compute_loss(inputs, targets):
    log_probs = torch.nn.functional.log_softmax(inputs, dim=1)
    selected = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
    return -selected.mean()

model = MNIST()

In [ ]:
model(x_train)

In [ ]:
class TrainDataset(Dataset):
    def __init__(self, features, targets):
        self.features = self._normalize(features)
        self.targets = targets

    def _normalize(self, data):
        return data / 255.0

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


class TestDataset(Dataset):
    def __init__(self, features, targets):
        self.features = self._normalize(features)
        self.targets = targets

    def _normalize(self, data):
        return data / 255.0

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


In [ ]:
train_ds = TrainDataset(features=x_train, targets=y_train)
test_ds = TestDataset(features=x_test, targets=y_test)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=64)

In [ ]:
for epoch in range(epochs):
    batch_loss = 0.0
    cnt = 0
    for xb, yb in train_dl:
        cnt += 1
        ypred = model(xb)
        
        loss = compute_loss(ypred, yb)
        batch_loss += loss.item()
        loss.backward()
        with torch.no_grad():
            model.W1 -= lr*model.W1.grad
            model.b1 -= lr*model.b1.grad
            model.W2 -= lr*model.W2.grad
            model.b2 -= lr*model.b2.grad
            model.W3 -= lr*model.W3.grad
            model.b3 -= lr*model.b3.grad
            model.W1.grad.zero_()
            model.b1.grad.zero_()
            model.W2.grad.zero_()
            model.b2.grad.zero_()
            model.W3.grad.zero_()
            model.b3.grad.zero_()
    print(batch_loss/cnt)